In [13]:
# IMPORTS

from classes.single_encoder import SingleEncoder
from classes.dual_encoder_single import DualEncoderAsSingle
from helpers.generate_embeddings import generate_embeddings
from helpers.load_embeddings import load_embeddings
from helpers.retrieve_top_k import retrieve_top_k
from helpers.load_dual_encoder import load_dual_encoder_model
from helpers.load_single_encoder import load_single_encoder_model
from helpers.retrieve_weight_samples import retrieve_weighted_samples
from transformers import CanineModel, CanineTokenizer
import pandas as pd
import torch
import faiss
import numpy as np
import pickle
import os
from openai import OpenAI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
# SETUP
API_KEY = ""
with open("API_KEY", "r") as f:
    API_KEY = f.read()
client = OpenAI(api_key=API_KEY)

MODEL_PATH = "../output/models/"
EMBEDDINGS_PATH = "../output/embeddings/"
CHAT_DATA_PATH = "../data/rag/"

# MODEL_NAME = "base_canine"
# MODEL_NAME = "author_contrastive"
MODEL_NAME = "style_discovery_loss"

CHAT_DATA = "private_embeddings_full"


MODEL_PATH = MODEL_PATH + MODEL_NAME
EMBEDDINGS_PATH = EMBEDDINGS_PATH + MODEL_NAME
CHAT_DATA_PATH = CHAT_DATA_PATH + CHAT_DATA + ".csv"

#=====================================
# LOAD MODEL
# Base CANINE
# tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
# encoder = CanineModel.from_pretrained("google/canine-s")
# model = SingleEncoder()
# model.encoder = encoder  # replace the encoder
# model.to(device)
# model.eval()

# Author Contrastive Loss
# model, tokenizer, device = load_single_encoder_model(MODEL_PATH)

# Style Discovery Loss
model, tokenizer, device = load_dual_encoder_model(MODEL_PATH)
model = DualEncoderAsSingle(model)

Loaded model weights from ../output/models/style_discovery_loss\dual_encoder.pt
Loaded projection heads from ../output/models/style_discovery_loss\projection_head.pt
Loaded tokenizer from ../output/models/style_discovery_loss
Model loaded successfully
Device: cuda
Model dir: ../output/models/style_discovery_loss


In [15]:
# LOAD CHAT DATA AND GENERATE EMBEDDINGS
df = pd.read_csv(CHAT_DATA_PATH)
df = df.dropna(subset=["Content"])
df["Content"] = df["Content"].astype(str)

print("Messages:", len(df))
print("Authors:", df["Author"].nunique())

texts = df["Content"].tolist()

embeddings = generate_embeddings(
    texts=texts,
    model=model,
    tokenizer=tokenizer,
    batch_size=8,
    device=device,
)

print("Embedding shape:", embeddings.shape)

records = [
    {
        "author": df.iloc[i]["Author"],
        "content": df.iloc[i]["Content"]
    }
    for i in range(len(df))
]

# Save to FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine similarity
index.add(embeddings.numpy())

print("Vectors in index:", index.ntotal)

os.makedirs(EMBEDDINGS_PATH, exist_ok=True)

faiss.write_index(index, os.path.join(EMBEDDINGS_PATH, "style_index.faiss"))
with open(os.path.join(EMBEDDINGS_PATH, "style_metadata.pkl"), "wb") as f:
    pickle.dump(records, f)
torch.save(embeddings, os.path.join(EMBEDDINGS_PATH, "style_embeddings.pt"))

Messages: 40713
Authors: 2


100%|██████████| 5090/5090 [02:18<00:00, 36.81it/s]


Embedding shape: torch.Size([40713, 64])
Vectors in index: 40713


In [16]:
# load embeddings directly if already generated
index, records, embeddings = load_embeddings(EMBEDDINGS_PATH)

In [17]:
# Example query
query_text = "✍️🔥🔥🔥🔥🔥"

# Retrieve weighted samples
retrieved_results = retrieve_weighted_samples(
    query_text=query_text,
    model=model,
    tokenizer=tokenizer,
    index=index,
    records=records,
    device=device,
    k=20,
    temperature=0.03  # Adjust for more/less diversity
)

# Print results
for i, result in enumerate(retrieved_results):
    print(f"\n--- Result {i+1} ---")
    print(f"Author: {result['author']}")
    print(f"Content: {result['content']}")
    print(f"Probability Rank: {result['rank_by_probability']}")


--- Result 1 ---
Author: fa09cb53
Content: freshmans when they have to communicate
Probability Rank: 4

--- Result 2 ---
Author: fa09cb53
Content: the only time when this is actualyl useful 😤
Probability Rank: 7

--- Result 3 ---
Author: e1688f3c
Content: r/foundthemobileuser fandom is getting downvoted a shit ton  right now
Probability Rank: 12

--- Result 4 ---
Author: fa09cb53
Content: this shit is... KINO!!!!
Probability Rank: 16

--- Result 5 ---
Author: e1688f3c
Content: they score shit out of 7 in this course
Probability Rank: 22

--- Result 6 ---
Author: e1688f3c
Content: apparently the government forces university students to take "national defense education"
Probability Rank: 35

--- Result 7 ---
Author: e1688f3c
Content: no!!!!!! oops club disbanded⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉⁉
Probability Rank: 37

--- Result 8 ---
Author: e1688f3c
Content: your parents planning on doing anything about this?
Probability Rank: 40

--- Result 9 ---
Author: e1688f3c
Content: maybe i'll put the lighting sh

In [18]:
# Generate response using OpenAI GPT model
def build_prompt(
    user_message,
    style_examples
):
    style_block = "\n".join(f"- {ex['content']}" for ex in style_examples)

    prompt = f"""You will receive a message as part of a text conversation.
You are going to give a reply suggestion to the latest message based on the writing style inferred from provided examples.

Based on the examples below, infer the writing style, such as tone, formality, common phrases, and slang.
Do NOT reuse topics, facts, names, or specific phrases from the examples.

Examples:
{style_block}

Write a short, 1-clause, sentence, non-specific reply to the message below, using ONLY the inferred style from the examples above.
Do not infer style from the message itself.
Do not make up any context in your response.
Do not add extra explanation or context outside of the response.
"""
    return prompt


def generate_responses(query_text, model_name, model, tokenizer, device, client, k=3, gpt_model="gpt-5-mini"):
    index, records, embeddings = load_embeddings("../output/embeddings/" + model_name)
    responses = []

    for i in range(k):
        retrieved_results = retrieve_weighted_samples(
            query_text=query_text,
            model=model,
            tokenizer=tokenizer,
            index=index,
            records=records,
            device=device,
            k=10,
            temperature=0.03  # Adjust for more/less diversity
        )
        
        response = client.responses.create(
            model=gpt_model,
            instructions=build_prompt(
                user_message=query_text,
                style_examples=retrieved_results
            ),
            input=query_text,
        )

        responses.append(response.output_text)
        
    return responses

In [19]:
from helpers.generate_messages_with_context import message_to_string_format
import json

with open(r'..\output\rag\selected_messages_with_context.json', 'r') as f:
    data = json.load(f)

messages = [
    "when's gaming?",
    "how is your uncle doing?",
    "am i ready to meet your mother?",
    "yo wsg",
    "@grok is this true?",
    "swag nation we rise",
    "ok buddy",
    "whole milk or full milk?",
    "what platform do you use for short form videos?",
    "evil gang",
    "i think im about to blow",
    "how about jerk mates duo",
    "i have been renting her for weeks",
    "I NEED SWAG",
    "i really like the NBA, htey are my favourite sport organization",
    "youre retarded",
    "nword hard r",
    "This is Rocket League!",
    "does your balls hang low, do they wobble to the flow, can you tie them in a know, can you tie it in a bow",
    "i support the state of israel :)",
]

for m in messages:
    print(generate_responses(m, MODEL_NAME, model, tokenizer, device, client, k=1)[0])


later maybe i'll hop on 😎
no clue fam
idk maybe later
bruh just vibin lowkey 😎
idk man sounds sketchy
let's goooo hype city!!!
bet i'm on it
idk man
depends bro
bruh that's wild
bruh chill fr 😬
sounds like a vibe…
that sounds sus af
you tryna stunt or nah?
i'm feeling that vibe bruh
bruh chill out
nah that's messed up
yo that's sick 😎🔥
bruh that's some next-level nonsense
respect bro
